# 03 — Model Evaluation
Evaluate the trained EfficientNet-B4 on the test set: confusion matrix, per-class F1, and inference speed.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
import time

from src.detection.model import load_model, DEFECT_CLASSES
from src.detection.dataset import FabricDefectDataset, get_val_transforms

MODEL_PATH = '../models/efficientnet/best.pt'
TEST_DIR = '../data/raw/test'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Load Model & Test Data

In [ ]:
model = load_model(MODEL_PATH, device)
test_ds = FabricDefectDataset(TEST_DIR, get_val_transforms(380))
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)
print(f'Test samples: {len(test_ds)}')

## Run Inference

In [ ]:
all_preds, all_labels = [], []
t0 = time.time()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

elapsed = time.time() - t0
fps = len(test_ds) / elapsed
print(f'Inference FPS: {fps:.1f} (target: >20)')

## Classification Report

In [ ]:
report = classification_report(all_labels, all_preds, target_names=DEFECT_CLASSES, output_dict=True)
df_report = pd.DataFrame(report).T
print(df_report.round(3).to_string())
print(f"\nOverall Accuracy: {report['accuracy']:.4f} (target: >0.92)")

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[c.replace('_', '\n') for c in DEFECT_CLASSES],
            yticklabels=[c.replace('_', '\n') for c in DEFECT_CLASSES], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Fabric Defect Classifier')
plt.tight_layout(); plt.show()

## Per-Class F1 Score

In [ ]:
f1_scores = {cls: report[cls]['f1-score'] for cls in DEFECT_CLASSES if cls in report}
df_f1 = pd.DataFrame(list(f1_scores.items()), columns=['Defect Class', 'F1 Score'])

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if v >= 0.80 else 'red' for v in df_f1['F1 Score']]
sns.barplot(data=df_f1.sort_values('F1 Score'), x='F1 Score', y='Defect Class', palette=colors, ax=ax)
ax.axvline(0.80, color='red', linestyle='--', label='Target F1=0.80')
ax.set_title('Per-Class F1 Score'); ax.legend()
plt.tight_layout(); plt.show()